## Task 2 — Response Selection

Multi-party conversation response selection with topic-aware attention (STP + MLM pretraining on DSTC-8 Ubuntu IRC).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, BertForMaskedLM
from torch.optim import AdamW
from tqdm import tqdm
import random, json

# ============================================================
# Dataset for Pretraining (STP + MLM)
# ============================================================
class TopicBERTPretrainDataset(Dataset):
    def __init__(self, convos, tokenizer, max_len=64, neg_ratio=4):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_len = max_len

        for convo in convos:
            msgs = [m["utterance"] for m in convo["messages-so-far"]]
            for i in range(len(msgs) - 1):
                u, v = msgs[i], msgs[i + 1]
                # positive pair
                self.samples.append((u, v, 1))
                # negatives: pair u with random utterances
                for _ in range(neg_ratio):
                    rand_v = random.choice(msgs)
                    if rand_v != v:
                        self.samples.append((u, rand_v, 0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def stp_collate_fn(batch, tokenizer, max_len=64, mlm_probability=0.15):
    us, vs, labels = zip(*batch)
    enc = tokenizer(
        list(us), list(vs),
        padding=True, truncation=True,
        max_length=max_len, return_tensors="pt"
    )

    labels = torch.tensor(labels)

    # MLM labels: mask tokens only for positives
    mlm_labels = enc["input_ids"].clone()
    mask = torch.full(mlm_labels.shape, -100)
    for i, y in enumerate(labels):
        if y == 1:  # only positives
            prob = torch.rand(mlm_labels[i].shape)
            mask_idx = prob < mlm_probability
            mlm_labels[i][~mask_idx] = -100  # keep only masked positions
        else:
            mlm_labels[i] = -100  # ignore negatives

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "token_type_ids": enc["token_type_ids"],
        "stp_labels": labels,
        "mlm_labels": mlm_labels,
    }

# ============================================================
# Topic-BERT with topic attention
# ============================================================
class TopicAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Ua = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, cls, tokens):
        cls_exp = self.Wa(cls).unsqueeze(1)
        tok_lin = self.Ua(tokens)
        e = self.v(torch.tanh(cls_exp + tok_lin)).squeeze(-1)
        a = torch.softmax(e, dim=-1)
        t_topic = torch.bmm(a.unsqueeze(1), tokens).squeeze(1)
        return t_topic


class TopicBERT(nn.Module):
    def __init__(self, model_name="bert-base-uncased", hidden_size=768):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.topic_att = TopicAttention(hidden_size)
        self.rs_head = nn.Linear(hidden_size * 2, 1)
        self.stp_head = nn.Linear(hidden_size * 2, 1)  # for pretraining

    def forward(self, input_ids, attention_mask, token_type_ids, B=None, M=None, task="rs"):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )
        cls = out.last_hidden_state[:, 0, :]
        tok = out.last_hidden_state
        t_topic = self.topic_att(cls, tok)
        t = torch.cat([cls, t_topic], dim=-1)  # [batch, 2H]

        if task == "stp":
            return self.stp_head(t).squeeze(-1), t
        elif task == "rs":
            # reshape for response selection
            N = input_ids.size(0) // (B * M)
            T = t.view(B, M, N, -1)
            pooled, _ = torch.max(T, dim=2)
            return self.rs_head(pooled).squeeze(-1), t
        else:
            raise ValueError("Unknown task")

# ============================================================
# Losses
# ============================================================
def compute_stp_loss(logits, labels):
    return F.binary_cross_entropy_with_logits(logits, labels.float())

def compute_mlm_loss(model_mlm, batch, device):
    outputs = model_mlm(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        labels=batch["mlm_labels"].to(device),
    )
    return outputs.loss

def compute_rs_loss(logits, labels):
    gold = labels.argmax(dim=1)
    return F.cross_entropy(logits, gold)

# ============================================================
# Training Functions
# ============================================================
def pretrain_one_epoch(model, model_mlm, loader, optimizer, device="cuda"):
    model.train(); model_mlm.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Pretraining"):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        # STP
        stp_logits, _ = model(batch["input_ids"], batch["attention_mask"], batch["token_type_ids"], task="stp")
        loss_stp = compute_stp_loss(stp_logits, batch["stp_labels"])

        # MLM
        loss_mlm = compute_mlm_loss(model_mlm, batch, device)

        loss = loss_stp + loss_mlm

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def train_one_epoch(model, loader, optimizer, device="cuda"):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Finetuning"):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        logits, _ = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch["token_type_ids"],
            batch["B"],
            batch["M"],
            task="rs",
        )
        loss = compute_rs_loss(logits, batch["labels"])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device="cuda"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
            logits, _ = model(
                batch["input_ids"],
                batch["attention_mask"],
                batch["token_type_ids"],
                batch["B"],
                batch["M"],
                task="rs",
            )
            preds = logits.argmax(dim=1)
            gold = batch["labels"].argmax(dim=1)
            correct += (preds == gold).sum().item()
            total += gold.size(0)
    return correct / total

# ============================================================
# Example Run
# ============================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    # load DSTC8 train/dev jsons
    # train_data = json.load(open("train.json"))
    # dev_data = json.load(open("dev.json"))
    # quick test
    train_data = [ {
        "data-split": "dev",
        "example-id": 1000007,
        "messages-so-far": [
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:30",
                "utterance": "how can i boost microphone volume? The volume is toooooo low"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_1",
                "time": "12:30",
                "utterance": "participant_0 , look for a microphone boost in alsamixer"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_2",
                "time": "12:30",
                "utterance": "participant_0 : type 'alsamixer' into terminal"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:31",
                "utterance": "how the heck do i use alsamixer? :P what is microphone ?"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:32",
                "utterance": "how do i choose volume on input participant_2 ?"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_2",
                "time": "12:33",
                "utterance": "participant_0 : arrow keys up and down"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:33",
                "utterance": "participant_2 , yes i understand that. But wich one of those things am i supposed to choose ?"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_2",
                "time": "12:33",
                "utterance": "participant_0 : you wanted input, right?"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:34",
                "utterance": "participant_2 , yes. But i there is no way i can turn that up. :S"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_2",
                "time": "12:34",
                "utterance": "participant_0 : press tab to go over to capture, then turn it up"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:34",
                "utterance": "aha :) thanks"
            }
        ],
        "options-for-correct-answers": [
            {
                "candidate-id": "af1d8ab6750244bad6a6ded3c44313fd",
                "speaker": "participant_2",
                "utterance": "np :)"
            }
        ],
        "options-for-next": [
            {
                "candidate-id": "70be3f88171c75ce825e9ff0b3963ca7",
                "speaker": "participant_1",
                "utterance": "!ubuntu-desktop"
            },
            {
                "candidate-id": "9a8afb971c94a124852ed622fdf4936e",
                "speaker": "participant_1",
                "utterance": "where had u been?"
            },
            {
                "candidate-id": "d12ea36e2ad846465f23e12297218ad0",
                "speaker": "participant_1",
                "utterance": "!fuse"
            },
            {
                "candidate-id": "13487472e29ad11030539fe20ca577ed",
                "speaker": "participant_2",
                "utterance": "participant_0 : Yes. (The 'y' parameter is implicit according to the man page.)"
            },
            {
                "candidate-id": "41d8f8a12c81a2343655253b93c88076",
                "speaker": "participant_2",
                "utterance": "also, should i be using the alternate installer, or the standard installer?"
            },
            {
                "candidate-id": "93a44c564d89b5f92621a0f390ddb0e8",
                "speaker": "participant_1",
                "utterance": "you dont use ubuntu but you sure know your way around the gnome crap"
            },
            {
                "candidate-id": "c325d21fb77806afc9430c86849d12cb",
                "speaker": "participant_1",
                "utterance": "If I set the time correctly in Linux, and restart the time is one hour ahead again."
            },
            {
                "candidate-id": "ef35fd7630e455d99437adf40facf007",
                "speaker": "participant_2",
                "utterance": "hello participant_0"
            },
            {
                "candidate-id": "eca7fcfb6dcce46373df054e79a16ad4",
                "speaker": "participant_2",
                "utterance": "participant_0 : yup.. it sez.. permission denied"
            },
            {
                "candidate-id": "c354d2565b618b2fd0127d5ead5d4f44",
                "speaker": "participant_2",
                "utterance": "so does mine and its my main language"
            },
            {
                "candidate-id": "7d46f3354609703dd98655492f2f85fc",
                "speaker": "participant_2",
                "utterance": "I rebooted, participant_0 . https://termbin.com/up1u"
            },
            {
                "candidate-id": "3865ae63001cf07044c524b91a5569e8",
                "speaker": "participant_1",
                "utterance": "sudo apt-get install torrentflux participant_0"
            },
            {
                "candidate-id": "3617268393fbda96ad73b287405be0da",
                "speaker": "participant_1",
                "utterance": "perl: Depends: perl-base (= 5.8.4-2ubuntu0.2) but 5.8.4-2ubuntu0.1 is installed"
            },
            {
                "candidate-id": "b36b100606783c76deed970fa3b4f474",
                "speaker": "participant_1",
                "utterance": "...and not nearly as obscufatable as perl..."
            },
            {
                "candidate-id": "4aea9b910d4032b3866ac48d031ff8e8",
                "speaker": "participant_2",
                "utterance": "I want replace ubuntu with another linux.so please give some advice"
            },
            {
                "candidate-id": "75da9f1a120b57a7410f543c776a15ee",
                "speaker": "participant_2",
                "utterance": "participant_1 wouldn't it have to be if on the console with same user it works fine?"
            },
            {
                "candidate-id": "8b431c8995876409ac79ca8dde040942",
                "speaker": "participant_2",
                "utterance": "participant_1 : so, it's an encoding issue"
            },
            {
                "candidate-id": "8c08edde20a2469f20962929538a309d",
                "speaker": "participant_1",
                "utterance": "::1 ip6-localhost ip6-loopback how do i change this as apache2 doesn't recoginze it and won't open localhost"
            },
            {
                "candidate-id": "444a21ed10ede3ccc7b679609eff839b",
                "speaker": "participant_1",
                "utterance": "the prob started recently?"
            },
            {
                "candidate-id": "5ae2e790d9732f785debc8557000abb0",
                "speaker": "participant_1",
                "utterance": "participant_0 , put in fstab"
            },
            {
                "candidate-id": "dbab63bcea91154ae160e1842673ef85",
                "speaker": "participant_2",
                "utterance": "just run 'vmware'"
            },
            {
                "candidate-id": "0d1510c9acf885d3bbabb96a3b32a086",
                "speaker": "participant_1",
                "utterance": "participant_2 : Well I assume you are the one I told to install xorg-edgers?"
            },
            {
                "candidate-id": "24efd6cc82f786e94b44112ebb611b37",
                "speaker": "participant_2",
                "utterance": "ville, that was not apt-get though"
            },
            {
                "candidate-id": "40d23233e0db833deb39f39d9c68daef",
                "speaker": "participant_1",
                "utterance": "thank you sir"
            },
            {
                "candidate-id": "745b9024ad61296c79f638e0e851fc72",
                "speaker": "participant_1",
                "utterance": "help"
            },
            {
                "candidate-id": "b48f6941e0ee260c9c9d9c352db04ffa",
                "speaker": "participant_1",
                "utterance": "Lucid Lynx... nice. I like Karmic Koala name the most, out of the names of currently available editions. Not sure if I like it more or less than Lucid Lynx"
            },
            {
                "candidate-id": "9c9a5b091aa63dbac1458dd536b7b8ed",
                "speaker": "participant_2",
                "utterance": "participant_0 : i guess its a old package to"
            },
            {
                "candidate-id": "5678aa6deb0aed195ee2eb8d2d062d9c",
                "speaker": "participant_1",
                "utterance": "any ideas on how to get mythgame to install, or would it really not work?"
            },
            {
                "candidate-id": "e6872d740ece737eac359fea31947c39",
                "speaker": "participant_2",
                "utterance": "participant_0 , how's doing?"
            },
            {
                "candidate-id": "defa0bf3321eb138d0080be10ea7677d",
                "speaker": "participant_1",
                "utterance": "participant_2 , I guess that's what happens when you automate things. You forget how they work. :-)"
            },
            {
                "candidate-id": "d20941f793f7ec9319e2813fd22f10f0",
                "speaker": "participant_1",
                "utterance": "Hi, I'm having trouble with the final stage of installation. Whenever I restart and select Ubuntu from the menu it has a black screen with a white rectangle that says I need to have 1600x1200@60hz for the video settings. I have double-checked these but I keep getting the error!"
            },
            {
                "candidate-id": "c67c2c0cd4c4a397977cc153fba03498",
                "speaker": "participant_1",
                "utterance": "how come firefox loads the page but it doesnt show the page"
            },
            {
                "candidate-id": "5b97ea65730497f01ec409aa1741c793",
                "speaker": "participant_1",
                "utterance": "i am have a sound card issue and i am using compaq presario CQ40, anyone here can help me?"
            },
            {
                "candidate-id": "29a7759a547ffcbc922d45bd3c813673",
                "speaker": "participant_2",
                "utterance": "participant_1 : exactly"
            },
            {
                "candidate-id": "9be95087a34a353cea3dccd9529f790f",
                "speaker": "participant_2",
                "utterance": "whats a good pop client for ubuntu?"
            },
            {
                "candidate-id": "073c0fa9c43e9f06cf29b5865e3a4b2d",
                "speaker": "participant_2",
                "utterance": "participant_0 : thats ok. I will reinstall everything as soon as I can"
            },
            {
                "candidate-id": "796ee84b9a90729633860ce1a6c018ec",
                "speaker": "participant_1",
                "utterance": "participant_0 : ohhh ok let me check that"
            },
            {
                "candidate-id": "6f92fb8b6fbc2e5d68827449bf3ae250",
                "speaker": "participant_2",
                "utterance": "participant_0 : not even the console?"
            },
            {
                "candidate-id": "c7847480b09e39987127a6b5c9db8420",
                "speaker": "participant_1",
                "utterance": "The default interface in Ubuntu 11.04 is !Unity. You can switch back to regular !Gnome by logging out and clicking on your user name, in the Session box at the bottom of the screen select Ubuntu Classic."
            },
            {
                "candidate-id": "1fa678d8f98086b354fbac1661dcb38d",
                "speaker": "participant_2",
                "utterance": "participant_1 , spammers will be shot on sight"
            },
            {
                "candidate-id": "e4be2284960125fb2ef6f2fb3e6bca80",
                "speaker": "participant_1",
                "utterance": "!info lubuntu-desktop"
            },
            {
                "candidate-id": "70d5d1fcf700de9efef878c6947d7f87",
                "speaker": "participant_1",
                "utterance": "participant_2 , just tried again, and yes i can get access to it now, must have been a temporary fault & its been fixed"
            },
            {
                "candidate-id": "076b6d96c3c7d47b7eafc084084b4698",
                "speaker": "participant_1",
                "utterance": "participant_2 i didnt say anything about dapper"
            },
            {
                "candidate-id": "de89d1a5626897399a1ed54d70c5f7b8",
                "speaker": "participant_2",
                "utterance": "but that's another battle altogether."
            },
            {
                "candidate-id": "673bee1970fe47d2df473f7cd019ac99",
                "speaker": "participant_1",
                "utterance": "heeeeeelp"
            },
            {
                "candidate-id": "5054735817217f5fb5c0b52dad383a75",
                "speaker": "participant_2",
                "utterance": "participant_0 : yes the one on the website was a_unreal_ sayin that there is a special iso for my mac in particular"
            },
            {
                "candidate-id": "11625c9e6fc64eb8561fc92ea51b31a7",
                "speaker": "participant_1",
                "utterance": "alright, so i know that the install for deb is sudo dpkg ..."
            },
            {
                "candidate-id": "6537cb26bece82228f37ff99786e185f",
                "speaker": "participant_1",
                "utterance": "participant_0 where is it ?"
            },
            {
                "candidate-id": "9709d4bf445f19e272d9bfb1fbc15258",
                "speaker": "participant_2",
                "utterance": "participant_1 : right I forgot you made this function in bashrc, not sure"
            },
            {
                "candidate-id": "3562d09989b8eb03e90d895bbab24f3a",
                "speaker": "participant_2",
                "utterance": "participant_1 , but that's just on my laptop, so your mileage may vary since i only have two slots i believe"
            },
            {
                "candidate-id": "3774f683e41b7a10ae9b13dacc2dfa0d",
                "speaker": "participant_2",
                "utterance": "thats the name in software center at least"
            },
            {
                "candidate-id": "b0e508b8e311135ff9d5a4557d14d687",
                "speaker": "participant_2",
                "utterance": "is mark shuttleworth ever on?"
            },
            {
                "candidate-id": "05f49919a89aa2b28f2dbde2eb10258a",
                "speaker": "participant_2",
                "utterance": "participant_0 : its kind of clunky"
            },
            {
                "candidate-id": "de383b217d2224bc5544c138ab4bff25",
                "speaker": "participant_1",
                "utterance": "Someone created a sources.list generator for Ubuntu but I'll have to go find it... :("
            },
            {
                "candidate-id": "63bc15f626422f818bd008d6474842eb",
                "speaker": "participant_2",
                "utterance": "i have a 8gb fat32 sandisk sansa i would like to defrag, is there any way i could do this?"
            },
            {
                "candidate-id": "5aef90473ef85c27eb85faf3342f48c9",
                "speaker": "participant_1",
                "utterance": "l"
            },
            {
                "candidate-id": "33beec23a4a60e7edb7de8a712031015",
                "speaker": "participant_2",
                "utterance": "participant_1 , did you receive the PM?"
            },
            {
                "candidate-id": "a2556e5940e02e14dac8bacbc18aec25",
                "speaker": "participant_2",
                "utterance": "participant_1 : so I guess I have to recreate the initrd with lvm support in it :/"
            },
            {
                "candidate-id": "273833a11242af9e04c13e87556721fc",
                "speaker": "participant_2",
                "utterance": "participant_0 : rt2570"
            },
            {
                "candidate-id": "e5db100d502881737bf957258b0a1c76",
                "speaker": "participant_2",
                "utterance": "participant_0 : pasted"
            },
            {
                "candidate-id": "e8510fac464152beaa28b42a9bb1d10f",
                "speaker": "participant_2",
                "utterance": "sp0: previous boot messages are part of the current dmesg file or /var/log/dmesg.0"
            },
            {
                "candidate-id": "79f2545034c190b1ba6d837e6e8d19f6",
                "speaker": "participant_2",
                "utterance": "participant_0 : nope... that wasn't it. : ("
            },
            {
                "candidate-id": "536997d39f65f55f2e4650f9123a4005",
                "speaker": "participant_1",
                "utterance": "participant_0 : or you could do an apt-get clean perhaps, but this will wipe all archives from /var/cache/apt/archives"
            },
            {
                "candidate-id": "2e98ab7537b900e108122dc0cd757756",
                "speaker": "participant_1",
                "utterance": "like that"
            },
            {
                "candidate-id": "b01a6f17c1e545c25589f362e5b1c381",
                "speaker": "participant_2",
                "utterance": "it isnt there"
            },
            {
                "candidate-id": "bb78f6f5b97ad6baac6c715a06094647",
                "speaker": "participant_2",
                "utterance": "it fails on the \"AUTOBUILD=1......\" line"
            },
            {
                "candidate-id": "3d2775d597c4902fb7df7c5b9354521f",
                "speaker": "participant_1",
                "utterance": "participant_2 : i think with an /etc/network/if-up.d/ script ?"
            },
            {
                "candidate-id": "d9d96e578bbb79e76a0025ff9c3c398b",
                "speaker": "participant_2",
                "utterance": "Hey guys"
            },
            {
                "candidate-id": "784743a49896b9affb9c1126014aeab5",
                "speaker": "participant_1",
                "utterance": "ah"
            },
            {
                "candidate-id": "f83cd7a3fc1f3cce521a0769e766b919",
                "speaker": "participant_1",
                "utterance": "participant_2 : btw, i'm not interested in the quality because for now it's just about providing the ppa... the goal is not at all to be added to the official repositories or something like this."
            },
            {
                "candidate-id": "972cebd108a3bcbba62c3f1cd294ef3d",
                "speaker": "participant_2",
                "utterance": "participant_0 : if you want to see what packages where installed/removed, try /var/log/dpkg.log"
            },
            {
                "candidate-id": "762f2ded25293d9b78b7d59e9d18bc3f",
                "speaker": "participant_2",
                "utterance": "I can't find him"
            },
            {
                "candidate-id": "bfa5dbde034d5d158cdd0230d3fa091b",
                "speaker": "participant_2",
                "utterance": "any clues about how bringing it back ???"
            },
            {
                "candidate-id": "7904b5f286a0c100ca81c1d129fb2359",
                "speaker": "participant_1",
                "utterance": "pidgin has worked well for me, i'm trying to convert our office so we can ditch the yahoo! client which is now causing severe problems for many people on a regular basis"
            },
            {
                "candidate-id": "a5e119e676debb1f4d7be207df4fac5a",
                "speaker": "participant_2",
                "utterance": "have been wondering already :p"
            },
            {
                "candidate-id": "15368299305c8b5c74656bbb7ce3b56c",
                "speaker": "participant_2",
                "utterance": "i cant give you more details, since I'm heading off now"
            },
            {
                "candidate-id": "0e1dd2c3fe7cd9fb7bf90c65ce85a768",
                "speaker": "participant_1",
                "utterance": "but it gives an odea of tempersture inside the case, when the fan rotates at high speed"
            },
            {
                "candidate-id": "49772d1341f3ec87db6de1a84ac2195d",
                "speaker": "participant_1",
                "utterance": "aha participant_0 thanks =)"
            },
            {
                "candidate-id": "7573485cffb9c705b6bbbe4cc569536a",
                "speaker": "participant_2",
                "utterance": "Really no one who knows about terminal servers in ubuntu or edubuntu?"
            },
            {
                "candidate-id": "50558afb7bcd2585cbe542ed375f5b8b",
                "speaker": "participant_1",
                "utterance": "participant_2 when I install Fedora first it uses the Fedora grub but only Fedora is in it. If I install Ubuntu first there is no grub menu at all"
            },
            {
                "candidate-id": "421dc462c10636e4ce9d40b65b79a4b3",
                "speaker": "participant_2",
                "utterance": "participant_1 , how to install the driver?"
            },
            {
                "candidate-id": "7c272909cfad388db3c49bd6aa7b6288",
                "speaker": "participant_2",
                "utterance": "participant_1 , you should be able too. You only need to modify ports if your serving something and go through a router"
            },
            {
                "candidate-id": "ea6d906298a3e25ec9729dd72af27061",
                "speaker": "participant_1",
                "utterance": "i was compiling xchat manually and cpp fault came"
            },
            {
                "candidate-id": "8207d58006d0a3e5e226390e45ab2570",
                "speaker": "participant_1",
                "utterance": "!noroot | participant_2"
            },
            {
                "candidate-id": "0f998d3f52b5637b172d8e077e8541df",
                "speaker": "participant_1",
                "utterance": "participant_0 : Is recovery mode a failsafe mode ? And what am I meant to do there ?"
            },
            {
                "candidate-id": "0a4816ff3b4137e931a8a7b8e910cf8b",
                "speaker": "participant_1",
                "utterance": "i have to reboot so often because of them"
            },
            {
                "candidate-id": "4d498265103911da521b082c0d2b3cf7",
                "speaker": "participant_2",
                "utterance": "participant_0 yeah i guess it is all listed there in the \"my passport\" (the brand of external hdd i have). i was just thrown off because only the win7 section displayed itself in nautilus"
            },
            {
                "candidate-id": "3db4f19f9f29144f5997c34426e5ad9f",
                "speaker": "participant_1",
                "utterance": "oh thank goodness for cairo dock"
            },
            {
                "candidate-id": "afa41e63612294867fcac7be42e9b7c0",
                "speaker": "participant_1",
                "utterance": "participant_0 , no, cant see there"
            },
            {
                "candidate-id": "c6d771c8c7b3fd3142b91cafa5d4c2fc",
                "speaker": "participant_2",
                "utterance": "hello all !!"
            },
            {
                "candidate-id": "7e95a1569623b47b6f2db631f156ca1b",
                "speaker": "participant_1",
                "utterance": "so no locate to the apt or apt-get deb?"
            },
            {
                "candidate-id": "af1d8ab6750244bad6a6ded3c44313fd",
                "speaker": "participant_2",
                "utterance": "np :)"
            },
            {
                "candidate-id": "c4d26b796802cfa69b531f3485e41c98",
                "speaker": "participant_1",
                "utterance": "participant_0 : I think you might need to keep the quotes enclosing the \"100\""
            },
            {
                "candidate-id": "48fdecd820aa8ec93978ce9ea10326fd",
                "speaker": "participant_2",
                "utterance": "participant_0 : no, i didn't, and i don't remember how"
            },
            {
                "candidate-id": "1630608eff58ebe5d1193e77498a0f90",
                "speaker": "participant_2",
                "utterance": "i want to change some modules"
            },
            {
                "candidate-id": "bb9f089ea70b9e78557403afeb46ab25",
                "speaker": "participant_2",
                "utterance": "participant_0 , after installation, at the login window, click on session, and choose kde"
            },
            {
                "candidate-id": "2f8d517f89a5045166cf7fe3c0c8ee40",
                "speaker": "participant_2",
                "utterance": "Is there a ubuntu server edition if I want to install it on my server? Or, what do you guys recommend?"
            },
            {
                "candidate-id": "9888a093661401e9db03316aeb6e2919",
                "speaker": "participant_2",
                "utterance": "participant_1 : that's really weird. I thought it was its own article."
            },
            {
                "candidate-id": "b19557ef8af3ee9c8050d384b811f00d",
                "speaker": "participant_2",
                "utterance": "ubntu but that is personal choice. I find ubuntu easier to use but exwindows users might like kde because it is similar"
            },
            {
                "candidate-id": "0a52c05443063d616628201eb0e1e0d6",
                "speaker": "participant_1",
                "utterance": "thats fucked"
            }
        ],
        "scenario": 1
    } ]  # put DSTC8 examples here
    dev_data = [ {
        "data-split": "dev",
        "example-id": 1000011,
        "messages-so-far": [
            {
                "date": "2007-03-02",
                "speaker": "participant_0",
                "time": "12:37",
                "utterance": "who can pastebin the normal ubuntu 6.10 sources.list for me pelase?"
            },
            {
                "date": "2007-03-02",
                "speaker": "participant_1",
                "time": "12:37",
                "utterance": "!sources | participant_0"
            },
            {
                "date": "2007-03-02",
                "speaker": "ubotu",
                "time": "12:37",
                "utterance": "participant_0 : The packages in Ubuntu are divided into several sections. More information at https://help.ubuntu.com/community/Repositories and http://www.ubuntu.com/ubuntu/components - See also !EasySource"
            }
        ],
        "options-for-correct-answers": [
            {
                "candidate-id": "e7b9cc1095a6732ff9d827f2cfc02a98",
                "speaker": "participant_1",
                "utterance": "!easysource | participant_0"
            }
        ],
        "options-for-next": [
            {
                "candidate-id": "7acd57c34f2a86dc3410dba2c8249a94",
                "speaker": "ubotu",
                "utterance": "participant_0 , the thing is i dont know what the module is. if you want to reboot until sound works you can then lsmod to figure out what module you need"
            },
            {
                "candidate-id": "f955a9452163331ca98a77b28aa8cd44",
                "speaker": "ubotu",
                "utterance": "participant_1 ; oh, nvm. that I know of, your talkin about the nvidia drivers whom are needed"
            },
            {
                "candidate-id": "27eff9ccc50c73956882022395172092",
                "speaker": "ubotu",
                "utterance": "participant_1 , I am trying to run adb"
            },
            {
                "candidate-id": "55aa3bbe7cb62e1847db344a44c540ef",
                "speaker": "participant_1",
                "utterance": "i put ect instead of etc"
            },
            {
                "candidate-id": "3800f09eaf961f54b0b23ec692ce7e05",
                "speaker": "ubotu",
                "utterance": "their 1300 people here cmon lol"
            },
            {
                "candidate-id": "2b98988c7a96feb37bc557632f3abdaa",
                "speaker": "participant_1",
                "utterance": "[ 165.134187] sd 6:0:0:0: [sdb] Attached SCSI removable disk"
            },
            {
                "candidate-id": "42ebd27fa8dd3f38daaf29fc445bb737",
                "speaker": "ubotu",
                "utterance": "just open run?"
            },
            {
                "candidate-id": "657dfbdb21bea1569085b29289e24087",
                "speaker": "participant_1",
                "utterance": "You're welcome! But keep in mind I'm just a bot ;-)"
            },
            {
                "candidate-id": "1cb3cf97dbeed55b95e37d1e8297683b",
                "speaker": "participant_1",
                "utterance": "participant_0 : yes"
            },
            {
                "candidate-id": "77131bf00265ab64cc0510bd1a260dab",
                "speaker": "participant_1",
                "utterance": "ubotu , i want crate my own packages it's formated as different for format deb,rpm files stored in my repo. it's a automated processs"
            },
            {
                "candidate-id": "55a3cf2c413737f7b02705d414aa99c2",
                "speaker": "participant_1",
                "utterance": "unfortunately, software properties is not opening"
            },
            {
                "candidate-id": "e63358b92d64c890dc99ca6a7058582e",
                "speaker": "ubotu",
                "utterance": "im installing ubuntu but I get an error: Error removing initramfs-tools | subprocess installed post-installation script returned error exit status 1"
            },
            {
                "candidate-id": "fe6f17cfe09ef07b8dc9d2345b2d5210",
                "speaker": "participant_1",
                "utterance": "well when linux kernel start up the version string show stability of ubuntu the protocol to other linux"
            },
            {
                "candidate-id": "058cebc6f1a2a53d3596169af06b2945",
                "speaker": "ubotu",
                "utterance": "name: #ubuntu? Really? From what I've seen on #debian they are total jerks. At least you can ASK things here."
            },
            {
                "candidate-id": "8fb232d7e738ef7e2ee1620b07a7db64",
                "speaker": "participant_1",
                "utterance": "that be grub 2"
            },
            {
                "candidate-id": "dbdb84706bcf0fbbf32ce52f5130ed6c",
                "speaker": "ubotu",
                "utterance": "how can i tell if i am using xaa or exa or whatever i might be using ?"
            },
            {
                "candidate-id": "c360b30d8976596b36a964a8388ddc3f",
                "speaker": "participant_1",
                "utterance": "I've \"kind of\" got other computers on the network"
            },
            {
                "candidate-id": "19bd75ddc4fe624a0e5272f6b4391129",
                "speaker": "ubotu",
                "utterance": "my pidgin crashes as soon as i join this chatroom!... can ne1 tellme whats goin wrong?"
            },
            {
                "candidate-id": "814247efccff0c7d75ba47a2e451f83b",
                "speaker": "participant_1",
                "utterance": "Forgot your password? Boot into recovery mode. What's the root password? See !sudo. Don't see *** in password prompts? That's normal. Sudo doesn't ask for your password? It remembers you for several minutes. Please use strong passwords, see https://help.ubuntu.com/community/StrongPasswords"
            },
            {
                "candidate-id": "ae062e7d9b3433222da194e1b052fab7",
                "speaker": "participant_1",
                "utterance": "Did anyone respond to me?"
            },
            {
                "candidate-id": "dde14f74538db729fce825dfbb2c9d5f",
                "speaker": "ubotu",
                "utterance": "participant_0 : effectively an off-screen buffer for applications to render to, before being copied to the display. See https://en.wikipedia.org/wiki/Compositing_window_manager"
            },
            {
                "candidate-id": "fc83fefd104e6aadd00c07e0386a523a",
                "speaker": "participant_1",
                "utterance": "participant_0 : what is included in that direcotyr that i make?"
            },
            {
                "candidate-id": "6836f089c56ada8efbe8d62428c566c5",
                "speaker": "participant_1",
                "utterance": "ubotu : then you have weakened your system security, well done"
            },
            {
                "candidate-id": "7f144b3cbe5f068833deb088fe800276",
                "speaker": "ubotu",
                "utterance": "participant_1 : when it works, its a cool idea/concept.., but when it hoses something, you might as well reinstall"
            },
            {
                "candidate-id": "fcddee8424a00d5e5ba6e9aec21724af",
                "speaker": "participant_1",
                "utterance": "hey, how can I determinate if a daemon is either start by systemd or sysinit ? I am running ubuntu 16.04"
            },
            {
                "candidate-id": "1d298ff4223fe11cfcbab4b8ac2fe4fe",
                "speaker": "ubotu",
                "utterance": "participant_0 : also, check out tovid.... it's a front end script in the reposositories that takes a ton of work out of the process as well."
            },
            {
                "candidate-id": "8306b63b5aac6dea4bf69decd3e19761",
                "speaker": "participant_1",
                "utterance": "ubotu : nothing happens, `which creator` gives nothing"
            },
            {
                "candidate-id": "c976c6e2e8806f06befce609eae658bd",
                "speaker": "ubotu",
                "utterance": "participant_0 : i don't know, maybe the envy install process blacklisted the fglrx driver. it might have been possible to still get it to work without xgl, but i'm not too versed in that turf"
            },
            {
                "candidate-id": "10c4f9a6785778c64ce1402310f2fcd1",
                "speaker": "participant_1",
                "utterance": "Geforce 2 GTS/PRO 64 MB suppose to work on compiz??"
            },
            {
                "candidate-id": "0b1b07fe01ca1063a1baf162a8e3e5dd",
                "speaker": "participant_1",
                "utterance": "participant_0 : maybe you need to install flac tools ?"
            },
            {
                "candidate-id": "6d1cda915fcd06a549d930c096e0014d",
                "speaker": "participant_1",
                "utterance": "ok thanks"
            },
            {
                "candidate-id": "a70dbe5300c1afae6df61921b20f2b34",
                "speaker": "participant_1",
                "utterance": "ubotu : do you have access to pastebin, should I pastebin too?"
            },
            {
                "candidate-id": "d4dae6a40cccfce1de423ded7e884ce5",
                "speaker": "participant_1",
                "utterance": "what vpn server are you using?"
            },
            {
                "candidate-id": "c0c37c3af9f56cea5698c957d4515727",
                "speaker": "participant_1",
                "utterance": "hmm... ntfs is the filesystem used in Windows NT and newer; to automatically mount your NTFS partition: https://wiki.ubuntu.com/AutomaticallyMountMSWindowsPartitions"
            },
            {
                "candidate-id": "c77d2a91795ecbe75f0bac97538ddaf1",
                "speaker": "ubotu",
                "utterance": "of course xterm may load up the default shell."
            },
            {
                "candidate-id": "baff7319bb941001f2e46f8420ad741a",
                "speaker": "ubotu",
                "utterance": "participant_1 : but i'm testing in a lxd of 16.04 and it works just fine to install .debs with ~ in them (via apt, which in turn is using dpkg). So `apt policy dpkg` ?"
            },
            {
                "candidate-id": "12545490e155d0280e9cf25f77d9c58c",
                "speaker": "participant_1",
                "utterance": "ubotu , sounds good"
            },
            {
                "candidate-id": "9479fdefb1abb7cdf747a1632d5a3000",
                "speaker": "ubotu",
                "utterance": "participant_1 depends which version"
            },
            {
                "candidate-id": "1f497af7a94afda958496b234d5055a8",
                "speaker": "participant_1",
                "utterance": "longwaver -k lemme see"
            },
            {
                "candidate-id": "83c97f90ced5009a779db3a857ccc3d6",
                "speaker": "ubotu",
                "utterance": "participant_1 : going to try something else real quick here, brb. Again, despite the result, I do appreciate the help."
            },
            {
                "candidate-id": "e7b9cc1095a6732ff9d827f2cfc02a98",
                "speaker": "participant_1",
                "utterance": "!easysource | participant_0"
            },
            {
                "candidate-id": "cce5fc2428b679dfe6b3fcd335a470a5",
                "speaker": "participant_1",
                "utterance": "ubotu : what od you want us to do about it ?"
            },
            {
                "candidate-id": "1862906bcece09c8c8ea452b2c933457",
                "speaker": "participant_1",
                "utterance": "ubotu : something with.. NOPASS. I think.."
            },
            {
                "candidate-id": "6bc5822c9d3e647d88480a73972c23c2",
                "speaker": "ubotu",
                "utterance": "participant_1 : network hacking is not in the focus of this channel"
            },
            {
                "candidate-id": "f34d690fab2840023050c1ea315511cf",
                "speaker": "participant_1",
                "utterance": "participant_0 , did the final version of ubuntu 10.10 come up?"
            },
            {
                "candidate-id": "8e9f1b6d6aa959eb2a01ad9452cfe88b",
                "speaker": "ubotu",
                "utterance": "dropped packages means high contention on the network, a faulty route, or a faulty link"
            },
            {
                "candidate-id": "0f57824938ef5031ace7d6250ec9b332",
                "speaker": "participant_1",
                "utterance": "ok participant_0 now im in GParted, how do i get to the partition part at the top, with windows its just Alt, but not here"
            },
            {
                "candidate-id": "cde6162587997f3b005b81332a2eb9aa",
                "speaker": "participant_1",
                "utterance": "hi... I have mono 2.10.8.1 and monodevelop 2.8.6.3 installed on my computer (im using ubuntu 12.04) when I want to compile my program which has to use System.Net, it does not recognize it. then i saw, under my folder /usr/lib/mono, i have directories of 2.0 3.5 and 4.0. I could find System.Net under directory 4.0, but i couldnt make use of it... does anyone know how i can solve this problem?"
            },
            {
                "candidate-id": "e315b15ebd5b336cee9ceb3b3bb5316e",
                "speaker": "ubotu",
                "utterance": "well I'm just going to try some of these things now. participant_1 , participant_1 , thanks bunches. If it doesn't work, you'll see me again."
            },
            {
                "candidate-id": "de552bb53a439f33ac4b5adca479b765",
                "speaker": "participant_1",
                "utterance": "participant_0 : whats your video card?"
            },
            {
                "candidate-id": "a8e8f3f029f559451fd94d5d20df4d89",
                "speaker": "participant_1",
                "utterance": "Hi there, I just got a new Logitech microphone and it seems to work but I have to be incredibly close to it to talk into it. Basically, it's very quiet in Ubuntu but it works fine in Windows. Any tips on fixing this?"
            },
            {
                "candidate-id": "82376b4cf79f2394d789a9ef1dba1056",
                "speaker": "ubotu",
                "utterance": "yw:-)"
            },
            {
                "candidate-id": "6c6eb8714296f766e8296b2c5ced14d9",
                "speaker": "ubotu",
                "utterance": "participant_1 , ok"
            },
            {
                "candidate-id": "a1598e36dd64407983d00068d917ed6a",
                "speaker": "participant_1",
                "utterance": "participant_0 : That is an odd default :/"
            },
            {
                "candidate-id": "4ecde85c6b59f40bc1ffefe624ac014b",
                "speaker": "ubotu",
                "utterance": "EasyUbuntu is a script that automates installation of some items. Use at your own risk. See http://easyubuntu.freecontrib.org/ ; for help and or discussions about EasyUbuntu please join #easyubuntu."
            },
            {
                "candidate-id": "42f43e9a23a8fca3eeef52756d86d918",
                "speaker": "ubotu",
                "utterance": "Does anyone tell me how to enable XDMC serving???"
            },
            {
                "candidate-id": "77ef6cbb543ef95c2b5df98c4568168e",
                "speaker": "ubotu",
                "utterance": "participant_0 , you need to install the samba server"
            },
            {
                "candidate-id": "e0e838d51aaace434e149dab8f80061d",
                "speaker": "participant_1",
                "utterance": "https://www.irccloud.com/pastebin/jBXZEMTe"
            },
            {
                "candidate-id": "824bf03532a52c23a87f1ce98b4f4350",
                "speaker": "participant_1",
                "utterance": "participant_0 : ok, that makes sense"
            },
            {
                "candidate-id": "d951665307a0166082facea2605b1e98",
                "speaker": "participant_1",
                "utterance": "just can't remember why"
            },
            {
                "candidate-id": "c48cbb49e8298e13b58bb7e07f221639",
                "speaker": "ubotu",
                "utterance": "first of all, I can't disable the session."
            },
            {
                "candidate-id": "8b1887e65ec0d359a97f8472210a06ac",
                "speaker": "ubotu",
                "utterance": "and by mistake I mounted it in ~"
            },
            {
                "candidate-id": "22e183e0e82ff72a1ee3a38be19b8218",
                "speaker": "ubotu",
                "utterance": "looks like the browser hung on something and windows terminated it."
            },
            {
                "candidate-id": "006d19f50c9afdae14e7bb49b454acd9",
                "speaker": "participant_1",
                "utterance": "again???"
            },
            {
                "candidate-id": "ecf335ecef4806f80a85b0e09f2468b3",
                "speaker": "ubotu",
                "utterance": "participant_0 , thx for all your help and time!"
            },
            {
                "candidate-id": "520b0dd15c278d98d4815aa4bcad0f39",
                "speaker": "ubotu",
                "utterance": "participant_0 : ask vmware support, this channel is for ubuntu support"
            },
            {
                "candidate-id": "9b3a01033bf1db47680e9fc3560a5844",
                "speaker": "participant_1",
                "utterance": "ubotu : Add the information to your .bashrc?"
            },
            {
                "candidate-id": "bd81fa85a082ba9d65a1400d7769ad48",
                "speaker": "ubotu",
                "utterance": "participant_0 , recognize it and read from it."
            },
            {
                "candidate-id": "689ec3334d22d567113159681dc0b6a7",
                "speaker": "ubotu",
                "utterance": "participant_1 is baffled"
            },
            {
                "candidate-id": "b5c7a9cfc88cc4ada5330a0bba96149f",
                "speaker": "ubotu",
                "utterance": "how can I keep it in the original form"
            },
            {
                "candidate-id": "b022a5b467959f402833837b496b9755",
                "speaker": "ubotu",
                "utterance": "participant_0 no"
            },
            {
                "candidate-id": "bc807943e8057ca193d2e669e1ab0e9e",
                "speaker": "participant_1",
                "utterance": "ubotu : no gui at all?"
            },
            {
                "candidate-id": "4d7fc747351a342020123e7d03cb6371",
                "speaker": "participant_1",
                "utterance": "I have some problems with mounting USB serial. It changes ttyUSB*. IS it possible to force it to mount to same ttyUSB or to make som symlink?"
            },
            {
                "candidate-id": "b58cf9ef95aee59023fe3226908a655f",
                "speaker": "ubotu",
                "utterance": "and you'll need to elaborate a bit more about the sharing files thing"
            },
            {
                "candidate-id": "564d9dae4c21a2aa8629338f325688a4",
                "speaker": "ubotu",
                "utterance": "participant_0 : that will add \"win\" to places"
            },
            {
                "candidate-id": "be03ed201760768820931e6f2afe060d",
                "speaker": "ubotu",
                "utterance": "samthewildone@Olympus:~$ sudo mk2fs.vfat /dev/sdb1"
            },
            {
                "candidate-id": "1fa0c2ed3ac6655564279a387792c101",
                "speaker": "ubotu",
                "utterance": "participant_1 , not sure. I think it's because the installer didn't know where to put it?"
            },
            {
                "candidate-id": "15b2a1ac4c5d407d0418b933282e1647",
                "speaker": "ubotu",
                "utterance": "participant_0 , we support ubuntu here"
            },
            {
                "candidate-id": "c23441d3d26a20448cdaa2759faaee02",
                "speaker": "participant_1",
                "utterance": "UBUNTU!!"
            },
            {
                "candidate-id": "14488d12cc367fef5b71ab6c50dfe517",
                "speaker": "participant_1",
                "utterance": "ubotu , I really find it weird.. tell me, once you hear the startup sound from Ubuntu, have you tried doing Ctrl+Alt+F1?"
            },
            {
                "candidate-id": "df776a7057c2fbc45e5aa1705e522c18",
                "speaker": "participant_1",
                "utterance": "ubotu : what was what for? if you're hoping for a solution to your problem, state the details and if anyone knows, they'll answer"
            },
            {
                "candidate-id": "52b742ae66ba15bb7f0023bdaca932a7",
                "speaker": "participant_1",
                "utterance": "@ ubotu - here is the pastebin for the memory and cpuinfo"
            },
            {
                "candidate-id": "558ee5dc14c061ee594a75d84b4e5b99",
                "speaker": "ubotu",
                "utterance": "QT, does it come with ubuntu?"
            },
            {
                "candidate-id": "37daa9fde3295d45a82ddaa1f732ff7d",
                "speaker": "participant_1",
                "utterance": "Hello! Can I install Gnome 3 on Ubuntu 11.10? Are there any bugs or is it a straightforward process?"
            },
            {
                "candidate-id": "3dd5b34e1db50e22df908d1a4c9926ba",
                "speaker": "ubotu",
                "utterance": "maverick: please use the #bash channel for scripting help"
            },
            {
                "candidate-id": "82adfdd4c8a220e1c5baf04623523786",
                "speaker": "ubotu",
                "utterance": "i mean"
            },
            {
                "candidate-id": "6f1103a928b7fd4a595e5efb405978e2",
                "speaker": "ubotu",
                "utterance": "participant_1 : unacceptatble bugs in Karmic?"
            },
            {
                "candidate-id": "8de1ee67ccff76a657d302a8066c2548",
                "speaker": "ubotu",
                "utterance": "participant_1 : oh, I wouldn't consider any unimportant, it's just that the teams have grown to the point that they need their own channels. A good thing"
            },
            {
                "candidate-id": "bca5db2f96806ef6d0dfc9ebbe3f4cd9",
                "speaker": "ubotu",
                "utterance": "participant_0 , only the default vesa drivers have ; before you do some thing to nvidia copy the xorg..and then..:))"
            },
            {
                "candidate-id": "737fc787355114392d370f8bec63fba3",
                "speaker": "ubotu",
                "utterance": "or that maybe he doesnt have widescreen support installed yet."
            },
            {
                "candidate-id": "f31f5f30784a6be59a6dbd3591e633f1",
                "speaker": "ubotu",
                "utterance": "participant_1 how will that work when he doesn't know his password :-)"
            },
            {
                "candidate-id": "ad010b528f268661c6fd77866dae570f",
                "speaker": "ubotu",
                "utterance": "participant_1 : Does it all look like a network driver issue?"
            },
            {
                "candidate-id": "8804b9dbab08902db080c60776c48ea6",
                "speaker": "ubotu",
                "utterance": "the language packs were upgraded and i dont know which keyboard input method system to select"
            },
            {
                "candidate-id": "a54a6087043a6af0f63bdaf555789387",
                "speaker": "ubotu",
                "utterance": "participant_1 : Exactly, so you can find the hungarian name and change it or declare a different default language for it."
            },
            {
                "candidate-id": "d08283077cbacf96cc3a9ab820b8cd95",
                "speaker": "participant_1",
                "utterance": "ubotu so choose /dev/sdc3 specifically?"
            },
            {
                "candidate-id": "29e6534f8179859bdf9cc3783e74c413",
                "speaker": "participant_1",
                "utterance": "it says: /dev/sda2 cannot allocate memory"
            },
            {
                "candidate-id": "f509dea9fb00613f43b0704396240bed",
                "speaker": "ubotu",
                "utterance": "?"
            },
            {
                "candidate-id": "c10dd85eb57552aaca8b2aa187624324",
                "speaker": "ubotu",
                "utterance": "participant_1 please help with missing or participant_1 installation"
            },
            {
                "candidate-id": "dd227663bee3f87d2b3adb92ee6f3e5b",
                "speaker": "ubotu",
                "utterance": "participant_1 , i have warty on my desktop and it rocks"
            },
            {
                "candidate-id": "5f0c98dbbe77bd6808083aeead88d1e5",
                "speaker": "participant_1",
                "utterance": "5d52e926856edf1ac1c6d631d460cb41"
            }
        ],
        "scenario": 1
    } ]

    # -------- Pretraining dataset --------
    pretrain_dataset = TopicBERTPretrainDataset(train_data, tokenizer)
    pretrain_loader = DataLoader(
        pretrain_dataset, batch_size=16, shuffle=True,
        collate_fn=lambda b: stp_collate_fn(b, tokenizer)
    )

    # -------- Fine-tuning dataset --------
    class ResponseSelectionDataset(Dataset):
        def __init__(self, data, tokenizer, max_len=128):
            self.data, self.tokenizer, self.max_len = data, tokenizer, max_len
        def __len__(self): return len(self.data)
        def __getitem__(self, idx):
            ex = self.data[idx]
            ctx_utts = [m["utterance"] for m in ex["messages-so-far"]]
            cands = [c["utterance"] for c in ex["options-for-next"]]
            labels = [1 if c in [a["utterance"] for a in ex["options-for-correct-answers"]] else 0 for c in cands]
            return {"context": ctx_utts, "candidates": cands, "labels": labels, "id": ex["example-id"]}

    def rs_collate_fn(batch, tokenizer, max_len=128):
        contexts, candidates, labels, ids = [], [], [], []
        for b in batch:
            contexts.append(b["context"]); candidates.append(b["candidates"])
            labels.append(b["labels"]); ids.append(b["id"])
        B, M = len(contexts), len(candidates[0])
        pairs = []
        for ctx, cands in zip(contexts, candidates):
            for cand in cands:
                for utt in ctx:
                    pairs.append((utt, cand))
        enc = tokenizer([u for u, c in pairs],[c for u, c in pairs],
                        padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"],"attention_mask": enc["attention_mask"],
                "token_type_ids": enc["token_type_ids"],"labels": torch.tensor(labels),
                "B": B,"M": M,"ids": ids}

    train_dataset = ResponseSelectionDataset(train_data, tokenizer)
    dev_dataset = ResponseSelectionDataset(dev_data, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                              collate_fn=lambda b: rs_collate_fn(b, tokenizer))
    dev_loader = DataLoader(dev_dataset, batch_size=2,
                            collate_fn=lambda b: rs_collate_fn(b, tokenizer))

    # -------- Models --------
    model = TopicBERT().to(device)
    model_mlm = BertForMaskedLM.from_pretrained("bert-base-uncased").to(device)
    optimizer = AdamW(list(model.parameters()) + list(model_mlm.parameters()), lr=2e-5)

    # -------- Pretrain --------
    for epoch in range(2):
        loss = pretrain_one_epoch(model, model_mlm, pretrain_loader, optimizer, device)
        print(f"Pretrain Epoch {epoch}: loss={loss:.4f}")

    # -------- Fine-tune --------
    optimizer = AdamW(model.parameters(), lr=2e-5)  # fresh optimizer
    for epoch in range(3):
        loss = train_one_epoch(model, train_loader, optimizer, device)
        acc = evaluate(model, dev_loader, device)
        print(f"Finetune Epoch {epoch}: loss={loss:.4f}, Recall@1={acc:.4f}")


## Task 4 — Conversation Disentanglement

Thread disentanglement using `DisentangleHead` with features `[t_r ; t_k ; t_r ⊙ t_k ; t_r − t_k]`.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
from typing import List, Dict

# ======================================
# 1️⃣ Dataset Parser (DSTC8 Subtask 4)
# ======================================

def load_subtask4_convo(base_path: str, date_prefix: str):
    """
    Loads messages and annotations for one day of IRC logs.
    """
    tok_file = os.path.join(base_path, f"{date_prefix}.tok.txt")
    ann_file = os.path.join(base_path, f"{date_prefix}.annotation.txt")

    with open(tok_file, "r", encoding="utf-8") as f:
        messages = [line.strip() for line in f]

    annotations = {}
    with open(ann_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            parent, child = int(parts[0]), int(parts[1])
            annotations.setdefault(child, []).append(parent)
    return messages, annotations


# ======================================
# 2️⃣ Dataset Builder for Disentanglement
# ======================================

class DisentangleDataset(Dataset):
    """
    Builds pairs of (parent, child) utterances and binary label:
        1 if parent-child linked (same topic)
        0 if not linked
    """

    def __init__(self, msgs: List[str], annotations: Dict[int, List[int]], tokenizer,
                 max_len=128, window=100, min_index=1000):
        self.msgs = msgs
        self.ann = annotations
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.window = window
        self.min_index = min_index
        self.samples = []

        for i in range(len(msgs)):
            if i < min_index:
                continue
            start = max(0, i - window)
            candidates = list(range(start, i))
            true_parents = set(annotations.get(i, []))

            for c in candidates:
                label = 1 if c in true_parents else 0
                self.samples.append((msgs[c], msgs[i], label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        parent, child, label = self.samples[idx]
        enc = self.tokenizer(
            parent, child,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        return enc, torch.tensor(label, dtype=torch.float)


# ======================================
# 3️⃣ Model: Topic-BERT Encoder + Head
# ======================================

class TopicBERTEncoder(nn.Module):
    """
    Simplified Topic-BERT encoder from the paper.
    (Task 2 pretraining version feeds into this in the real paper)
    """

    def __init__(self, name="bert-base-uncased"):
        super().__init__()
        self.bert = BertModel.from_pretrained(name)
        self.proj = nn.Linear(self.bert.config.hidden_size, self.bert.config.hidden_size)
        self.out_dim = self.bert.config.hidden_size

    def forward(self, input_ids, attention_mask, token_type_ids):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        cls = out.last_hidden_state[:, 0, :]  # [CLS] vector
        t = torch.tanh(self.proj(cls))        # topic-enhanced embedding (like t_k or t_r)
        return t


class DisentangleHead(nn.Module):
    """
    Computes similarity features [t_r; t_k; t_r ⊙ t_k; t_r - t_k]
    and predicts if both utterances are same topic.
    """

    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, t_r, t_k):
        # compute feature vector
        f = torch.cat([t_r, t_k, t_r * t_k, t_r - t_k], dim=-1)
        logit = self.fc(f)
        return logit.squeeze(-1)


class TopicDisentanglementModel(nn.Module):
    """
    Full Task 4 Model = Shared Encoder + Binary Classifier
    """

    def __init__(self, bert_name="bert-base-uncased"):
        super().__init__()
        self.encoder = TopicBERTEncoder(bert_name)
        self.head = DisentangleHead(self.encoder.out_dim)

    def forward(self, input_ids, attention_mask, token_type_ids):
        # Encode the pair
        t = self.encoder(input_ids, attention_mask, token_type_ids)
        # Split t into two halves to simulate (parent, child)
        # In dataset we encode each pair (parent, child) together, so CLS = joint rep.
        # Here we just feed t twice to head (for demonstration consistency)
        # In full paper they separate t_k (parent) and t_r (child).
        logits = self.head(t, t)
        return logits


# ======================================
# 4️⃣ Training / Evaluation
# ======================================

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    loss_fn = nn.BCEWithLogitsLoss()
    total_loss = 0

    for step, (batch, labels) in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        ttype = batch["token_type_ids"].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn, ttype)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if step % 100 == 0:
            print(f"Step {step} - Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)


def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch, labels in dataloader:
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            ttype = batch["token_type_ids"].to(device)
            labels = labels.to(device)

            logits = model(input_ids, attn, ttype)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Validation Accuracy: {acc:.4f}")
    return acc


# ======================================
# 5️⃣ Example Usage
# ======================================

if __name__ == "__main__":
    base_dir = "/path/to/train_folder"
    date_prefix = "2007-03-02"  # example file set

    messages, annotations = load_subtask4_convo(base_dir, date_prefix)
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    dataset = DisentangleDataset(messages, annotations, tokenizer)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = TopicDisentanglementModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    for epoch in range(2):
        print(f"Epoch {epoch+1}")
        loss = train_epoch(model, dataloader, optimizer, device)
        evaluate(model, dataloader, device)
